In [21]:
room = [
    [1, 1, 1, 0],
    [0, -1, 2, 1],
    [1, 0, -1, 1]
]

rows = len(room)
cols = len(room[0])

In [22]:
def find_robot(state):
    for r in range(rows):
        for c in range(cols):
            if state[r][c] in [2, 3]:
                return (r, c)
    return None

In [23]:
def goal_test(state):
    for row in state:
        if 1 in row or 3 in row:
            return False
    return True

In [24]:
def copy_state(state):
    return [row[:] for row in state]

In [25]:
def count_dirty_tiles(state):
    # Đếm số ô rác
    count = 0
    for row in state:
        for tile in row:
            if tile in [1, 3]:
                count += 1
    return count

In [26]:
def actions(state):
    r, c = find_robot(state)
    moves = []

    if r > 0 and state[r-1][c] != -1: moves.append("Up")
    if r < rows - 1 and state[r+1][c] != -1: moves.append("Down")
    if c > 0 and state[r][c-1] != -1: moves.append("Left")
    if c < cols - 1 and state[r][c+1] != -1: moves.append("Right")

    if state[r][c] == 3:
        moves.append("Suck")
    return moves

In [27]:
def child_node(parent, action):
    state = parent["state"]
    new_state = copy_state(state)
    r, c = find_robot(state)
    
    if action == "Suck":
        new_state[r][c] = 2
    else:

        new_state[r][c] = 1 if state[r][c] == 3 else 0
        
        nr, nc = r, c
        if action == "Up": nr -= 1
        elif action == "Down": nr += 1
        elif action == "Left": nc -= 1
        elif action == "Right": nc += 1
        
        new_state[nr][nc] = 3 if new_state[nr][nc] == 1 else 2

    return {
        "state": new_state,
        "parent": parent,
        "action": action,
        "cost": parent["cost"] + 1
    }

In [28]:
def solution(node):
    path = []
    while node["parent"] is not None:
        path.append(node["action"])
        node = node["parent"]
    path.reverse()
    return path

In [29]:
import heapq

def greedy_best_first_search(initial_state):
    count = 0 

    h_n = count_dirty_tiles(initial_state)
    node = {
        "state": initial_state, 
        "parent": None, 
        "action": None, 
        "cost": 0, 
        "id": count
    }
    
    frontier = []
    heapq.heappush(frontier, (h_n, node["id"], node))
    
    reached = {str(initial_state)}
    
    while frontier:

        h_val, node_id, node = heapq.heappop(frontier)
        

        if goal_test(node["state"]):
            return node
            
        for action in actions(node["state"]):
            child = child_node(node, action)
            s_str = str(child["state"])
            
            if s_str not in reached:
                reached.add(s_str)
                count += 1
                child["id"] = count

                h_child = count_dirty_tiles(child["state"])
                heapq.heappush(frontier, (h_child, child["id"], child))
                
    return None

In [30]:
def print_room(state):
    for row in state:
        print(row)
    print(f"Số ô rác còn lại: {count_dirty_tiles(state)}")
    print("-" * 20)

result_node = greedy_best_first_search(room)

if result_node:
    path = solution(result_node)
    print(f"Tìm thấy lời giải! Tổng chi phí: {result_node['cost']} bước.")
    
    temp_node = {"state": room, "parent": None, "action": None, "cost": 0}
    print("Trạng thái ban đầu:")
    print_room(room)
    
    for act in path:
        temp_node = child_node(temp_node, act)
        print(f"Hành động: {act}")
        print_room(temp_node["state"])
else:
    print("Không tìm thấy đường đi.")

Tìm thấy lời giải! Tổng chi phí: 18 bước.
Trạng thái ban đầu:
[1, 1, 1, 0]
[0, -1, 2, 1]
[1, 0, -1, 1]
Số ô rác còn lại: 6
--------------------
Hành động: Up
[1, 1, 3, 0]
[0, -1, 0, 1]
[1, 0, -1, 1]
Số ô rác còn lại: 6
--------------------
Hành động: Suck
[1, 1, 2, 0]
[0, -1, 0, 1]
[1, 0, -1, 1]
Số ô rác còn lại: 5
--------------------
Hành động: Left
[1, 3, 0, 0]
[0, -1, 0, 1]
[1, 0, -1, 1]
Số ô rác còn lại: 5
--------------------
Hành động: Suck
[1, 2, 0, 0]
[0, -1, 0, 1]
[1, 0, -1, 1]
Số ô rác còn lại: 4
--------------------
Hành động: Left
[3, 0, 0, 0]
[0, -1, 0, 1]
[1, 0, -1, 1]
Số ô rác còn lại: 4
--------------------
Hành động: Suck
[2, 0, 0, 0]
[0, -1, 0, 1]
[1, 0, -1, 1]
Số ô rác còn lại: 3
--------------------
Hành động: Down
[0, 0, 0, 0]
[2, -1, 0, 1]
[1, 0, -1, 1]
Số ô rác còn lại: 3
--------------------
Hành động: Down
[0, 0, 0, 0]
[0, -1, 0, 1]
[3, 0, -1, 1]
Số ô rác còn lại: 3
--------------------
Hành động: Suck
[0, 0, 0, 0]
[0, -1, 0, 1]
[2, 0, -1, 1]
Số ô rác còn lại: